# Amazon Fashion Collaborative Filtering Baseline (Scalable)

This notebook implements a **production-oriented collaborative filtering baseline** for Amazon Fashion using **implicit ALS** (latent-factor matrix factorization) and evaluates ranking quality with **HR@10**, **NDCG@10**, and **MRR@10** on a warm-start test set.


## Why This Algorithm

- **Library**: `implicit`
- **Algorithm**: `AlternatingLeastSquares` (ALS)

ALS is mathematically suited for large sparse user-item matrices because it factorizes interactions into low-rank user/item embeddings and optimizes by alternating over sparse normal equations. Runtime scales roughly with non-zero interactions (not with dense `#users x #items` storage), and memory stays sparse via CSR/COO matrices.


In [ ]:
# If needed (run once):
# %pip install implicit


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix, csr_matrix
from tqdm.auto import tqdm

try:
    from implicit.als import AlternatingLeastSquares
except ImportError as e:
    raise ImportError(
        "The 'implicit' library is required. Install it with: pip install implicit"
    ) from e


In [ ]:
@dataclass
class PipelineConfig:
    train_path: str = "research/mini_train.csv"
    test_path: str = "research/mini_test.csv"
    file_format: str = "csv"  # switch to "parquet" when needed

    user_col: str = "user_id"
    item_col: str = "parent_asin"
    rating_col: str = "rating"

    rating_max: float = 5.0
    positive_threshold: float = 4.0

    # implicit ALS
    factors: int = 128
    regularization: float = 0.05
    iterations: int = 20
    alpha: float = 40.0
    random_state: int = 42

    # ranking eval
    k: int = 10
    batch_size: int = 2048
    max_eval_users: Optional[int] = None  # set for quick debug runs


cfg = PipelineConfig()
cfg


In [ ]:
def read_interactions(path: str, file_format: str = "csv") -> pd.DataFrame:
    if file_format == "csv":
        return pd.read_csv(path)
    if file_format == "parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported file_format={file_format!r}. Use 'csv' or 'parquet'.")


def prepare_interactions(df: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:
    required = [cfg.user_col, cfg.item_col, cfg.rating_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    out = df[required].copy()
    out[cfg.rating_col] = pd.to_numeric(out[cfg.rating_col], errors="coerce")
    out = out.dropna(subset=required)

    # Keep one interaction per (user,item) to reduce redundant nnz entries.
    out = out.groupby([cfg.user_col, cfg.item_col], as_index=False)[cfg.rating_col].max()

    out[cfg.user_col] = out[cfg.user_col].astype(str)
    out[cfg.item_col] = out[cfg.item_col].astype(str)
    out[cfg.rating_col] = out[cfg.rating_col].astype(np.float32)
    return out


def build_mappings(train_df: pd.DataFrame, cfg: PipelineConfig):
    user_index = pd.Index(train_df[cfg.user_col].unique(), dtype="object")
    item_index = pd.Index(train_df[cfg.item_col].unique(), dtype="object")

    user_to_idx = {u: i for i, u in enumerate(user_index)}
    item_to_idx = {it: i for i, it in enumerate(item_index)}

    return user_to_idx, item_to_idx, user_index, item_index


def encode_interactions(
    df: pd.DataFrame,
    cfg: PipelineConfig,
    user_to_idx: Dict[str, int],
    item_to_idx: Dict[str, int],
    drop_unknown: bool = True,
) -> pd.DataFrame:
    out = df.copy()
    out["user_idx"] = out[cfg.user_col].map(user_to_idx)
    out["item_idx"] = out[cfg.item_col].map(item_to_idx)

    if drop_unknown:
        out = out.dropna(subset=["user_idx", "item_idx"])

    out["user_idx"] = out["user_idx"].astype(np.int32)
    out["item_idx"] = out["item_idx"].astype(np.int32)
    out[cfg.rating_col] = out[cfg.rating_col].astype(np.float32)
    return out


def build_sparse_matrices(
    encoded_train: pd.DataFrame,
    n_users: int,
    n_items: int,
    cfg: PipelineConfig,
) -> Tuple[csr_matrix, csr_matrix]:
    # Hu et al. style confidence weighting for implicit ALS.
    ratings = encoded_train[cfg.rating_col].to_numpy(dtype=np.float32)
    confidence = 1.0 + cfg.alpha * (ratings / cfg.rating_max)

    rows = encoded_train["user_idx"].to_numpy(dtype=np.int32)
    cols = encoded_train["item_idx"].to_numpy(dtype=np.int32)

    user_items = coo_matrix(
        (confidence, (rows, cols)),
        shape=(n_users, n_items),
        dtype=np.float32,
    ).tocsr()

    item_users = user_items.T.tocsr()
    return user_items, item_users


def train_als(item_users: csr_matrix, cfg: PipelineConfig) -> AlternatingLeastSquares:
    model = AlternatingLeastSquares(
        factors=cfg.factors,
        regularization=cfg.regularization,
        iterations=cfg.iterations,
        random_state=cfg.random_state,
    )
    model.fit(item_users, show_progress=True)
    return model


def _ideal_dcg(num_relevant: int, k: int) -> float:
    upto = min(num_relevant, k)
    if upto <= 0:
        return 0.0
    ranks = np.arange(1, upto + 1)
    return float(np.sum(1.0 / np.log2(ranks + 1)))


def evaluate_ranking(
    model: AlternatingLeastSquares,
    user_items: csr_matrix,
    encoded_test: pd.DataFrame,
    cfg: PipelineConfig,
) -> Dict[str, float]:
    positives = encoded_test[encoded_test[cfg.rating_col] >= cfg.positive_threshold]
    user_to_relevant = positives.groupby("user_idx")["item_idx"].agg(set).to_dict()

    eval_users = np.array(sorted(user_to_relevant.keys()), dtype=np.int32)
    if cfg.max_eval_users is not None:
        eval_users = eval_users[: cfg.max_eval_users]

    hr_scores: List[float] = []
    ndcg_scores: List[float] = []
    mrr_scores: List[float] = []

    for start in tqdm(range(0, len(eval_users), cfg.batch_size), desc="Evaluating"):
        batch_users = eval_users[start : start + cfg.batch_size]
        batch_user_items = user_items[batch_users]

        try:
            rec_items, _ = model.recommend(
                userid=batch_users,
                user_items=batch_user_items,
                N=cfg.k,
                filter_already_liked_items=True,
            )
            if rec_items.ndim == 1:
                rec_items = rec_items.reshape(1, -1)
        except Exception:
            # Fallback for implicit versions that do not support batch recommend.
            rec_list = []
            for u in batch_users:
                ids, _scores = model.recommend(
                    userid=int(u),
                    user_items=user_items[int(u)],
                    N=cfg.k,
                    filter_already_liked_items=True,
                )
                rec_list.append(ids)
            rec_items = np.vstack(rec_list)

        for row_id, u in enumerate(batch_users):
            recommended = [int(i) for i in rec_items[row_id].tolist() if int(i) >= 0]
            relevant = user_to_relevant.get(int(u), set())

            hit_ranks = [rank for rank, item_idx in enumerate(recommended, start=1) if item_idx in relevant]

            hr_scores.append(1.0 if hit_ranks else 0.0)
            mrr_scores.append((1.0 / hit_ranks[0]) if hit_ranks else 0.0)

            dcg = 0.0
            for rank, item_idx in enumerate(recommended, start=1):
                if item_idx in relevant:
                    dcg += 1.0 / np.log2(rank + 1)

            idcg = _ideal_dcg(len(relevant), cfg.k)
            ndcg_scores.append((dcg / idcg) if idcg > 0 else 0.0)

    if len(eval_users) == 0:
        return {f"HR@{cfg.k}": 0.0, f"NDCG@{cfg.k}": 0.0, f"MRR@{cfg.k}": 0.0, "eval_users": 0}

    return {
        f"HR@{cfg.k}": float(np.mean(hr_scores)),
        f"NDCG@{cfg.k}": float(np.mean(ndcg_scores)),
        f"MRR@{cfg.k}": float(np.mean(mrr_scores)),
        "eval_users": int(len(eval_users)),
    }


def recommend_for_user(
    model: AlternatingLeastSquares,
    user_id: str,
    user_to_idx: Dict[str, int],
    idx_to_item: pd.Index,
    user_items: csr_matrix,
    k: int = 10,
) -> pd.DataFrame:
    if user_id not in user_to_idx:
        return pd.DataFrame(columns=["parent_asin", "score"])

    u = user_to_idx[user_id]
    item_ids, scores = model.recommend(
        userid=int(u),
        user_items=user_items[int(u)],
        N=k,
        filter_already_liked_items=True,
    )

    return pd.DataFrame(
        {
            "parent_asin": [idx_to_item[i] for i in item_ids],
            "score": scores,
        }
    )


def run_pipeline(cfg: PipelineConfig):
    train_raw = read_interactions(cfg.train_path, cfg.file_format)
    test_raw = read_interactions(cfg.test_path, cfg.file_format)

    train_df = prepare_interactions(train_raw, cfg)
    test_df = prepare_interactions(test_raw, cfg)

    user_to_idx, item_to_idx, idx_to_user, idx_to_item = build_mappings(train_df, cfg)

    train_enc = encode_interactions(train_df, cfg, user_to_idx, item_to_idx, drop_unknown=True)
    test_enc = encode_interactions(test_df, cfg, user_to_idx, item_to_idx, drop_unknown=True)

    n_users = len(idx_to_user)
    n_items = len(idx_to_item)

    user_items, item_users = build_sparse_matrices(train_enc, n_users, n_items, cfg)

    model = train_als(item_users, cfg)
    metrics = evaluate_ranking(model, user_items, test_enc, cfg)

    artifacts = {
        "model": model,
        "user_items": user_items,
        "train_df": train_df,
        "test_df": test_df,
        "train_enc": train_enc,
        "test_enc": test_enc,
        "user_to_idx": user_to_idx,
        "item_to_idx": item_to_idx,
        "idx_to_user": idx_to_user,
        "idx_to_item": idx_to_item,
        "metrics": metrics,
    }
    return artifacts


In [ ]:
artifacts = run_pipeline(cfg)
artifacts["metrics"]


In [ ]:
# Example recommendations for 3 warm users from train
sample_users = artifacts["idx_to_user"][:3].tolist()
for uid in sample_users:
    print(f"\nUser: {uid}")
    display(
        recommend_for_user(
            model=artifacts["model"],
            user_id=uid,
            user_to_idx=artifacts["user_to_idx"],
            idx_to_item=artifacts["idx_to_item"],
            user_items=artifacts["user_items"],
            k=cfg.k,
        )
    )


## Production Notes

- To switch to Parquet for large-scale runs, set `cfg.file_format = "parquet"` and point paths to parquet files.
- Keep all sparse matrices in `float32` + `int32` index space to reduce memory pressure.
- Increase `factors` and `iterations` gradually after baseline validation.
- If full evaluation on every warm user is too slow, tune `batch_size` and/or temporarily use `max_eval_users` for faster experimentation.
